# Interband $\varepsilon_2$ of Gold — Rosei Model

Reproduction of the theoretical $\varepsilon_2$ from **Fig. 8** of Guerrisi, Rosei & Winsemius (1975).

**What is computed.** Rosei's Eq. (8):
$$\varepsilon_2(\omega) = \frac{4\pi^2 e^2}{3m_e^2\omega^2}\Big[|P_X|^2\, N_X\, J_X(\hbar\omega,T) \;+\; |P_L|^2\, N_L\, J_L(\hbar\omega,T)\Big]$$

where each JDOS integral $J_P$ (his Eq. 7) is a 1D integral over conduction-band energy $\mathcal{E}$:

$$J_P(\hbar\omega,T) = \int_{\mathcal{E}_{\min}^P}^{\mathcal{E}_{\max}^P} \frac{[1-f(\mathcal{E},\,T)]}{\sqrt{g^P(\mathcal{E})}} \; d\mathcal{E}$$

The factor $[1-f(\mathcal{E})]$ is the probability the final sp-band state is empty. The initial d-band state is always occupied ($f \approx 1$, dropped).

| | X (saddle) | L (ellipsoid) |
|:--|:--|:--|
| Singularity $1/\sqrt{g}$ | $1/\sqrt{\mathcal{E}_{\max} - \mathcal{E}}$ | $1/\sqrt{\mathcal{E} - \mathcal{E}_{\min}}$ |
| $\bar{B}$ | $B_v - B_c$ | $B_c + B_v$ |
| Multiplicity | $N_X = 6$ | $N_L = 8$ |

**Lifetime broadening.** The bare JDOS has a hard onset at $\mathcal{E}_g$. To account for the finite lifetime of electronic states (electron-electron, electron-phonon scattering, disorder), we convolve $\varepsilon_2(\omega)$ with a Lorentzian of HWHM $\Gamma$:

$$\varepsilon_2^{\text{broad}}(\omega) = \int \varepsilon_2(\omega') \; \frac{\Gamma/\pi}{(\omega - \omega')^2 + \Gamma^2}\; d\omega'$$

This is equivalent to replacing $\delta(\Delta - \hbar\omega) \to \frac{\Gamma/\pi}{(\Delta - \hbar\omega)^2 + \Gamma^2}$ in the original Fermi golden rule, i.e., giving each transition a Lorentzian spectral weight $\sim \hbar/\tau$ where $\tau$ is the quasiparticle lifetime.

In [4]:
import numpy as np
from scipy.integrate import quad
import matplotlib.pyplot as plt
from dataclasses import dataclass

kB = 8.617e-5   # Boltzmann constant [eV/K]
C  = 3.81       # hbar^2 / (2 m_e)  [eV Ang^2]

def fermi(E, T):
    """Fermi-Dirac distribution; E measured from E_F."""
    if T == 0:
        return 1.0 if E < 0 else (0.5 if E == 0 else 0.0)
    return 1.0 / (1.0 + np.exp(np.clip(E / (kB * T), -500, 500)))

# ── Band parameter classes ──────────────────────────────────────────────

@dataclass
class XPointParams:
    """X-point: E_c = E0c + Ac u - Bc v  (saddle)."""
    Ac: float; Bc: float; Av: float; Bv: float
    Eg: float; E0c: float = 0.0

    def __post_init__(self):
        self.Abar = self.Ac + self.Av
        self.Bbar = self.Bv - self.Bc           # saddle sign
        self.D    = self.Ac * self.Bv + self.Av * self.Bc

    def E_max(self, hw):  return self.E0c + (self.Ac / self.Abar) * (hw - self.Eg)
    def E_min(self, hw):  return self.E0c - (self.Bc / self.Bbar) * (hw - self.Eg)
    def prefactor(self):  return 1.0 / np.sqrt(self.Abar * abs(self.D))

@dataclass
class LPointParams:
    """L-point: E_c = E0c + Ac u + Bc v  (ellipsoidal minimum)."""
    Ac: float; Bc: float; Av: float; Bv: float
    Eg: float; E0c: float = 0.0

    def __post_init__(self):
        self.Abar = self.Ac + self.Av
        self.Bbar = self.Bc + self.Bv           # both positive
        self.D    = self.Ac * self.Bv - self.Bc * self.Av

    def E_min(self, hw):  return self.E0c + (self.Ac / self.Abar) * (hw - self.Eg)
    def E_max(self, hw):  return self.E0c + (self.Bc / self.Bbar) * (hw - self.Eg)
    def prefactor(self):  return 1.0 / np.sqrt(self.Abar * abs(self.D))

# ── Gold parameters (Christensen & Seraphin 1971) ───────────────────────

X = XPointParams(Ac=C/0.31, Bc=C/0.40, Av=C/0.19, Bv=C/0.15, Eg=1.94)
L = LPointParams(Ac=C/0.24, Bc=C/0.12, Av=C/0.70, Bv=C/1.03, Eg=2.45)

N_X, N_L   = 6, 8       # valley multiplicities
P_RATIO_SQ = 0.370      # |P_X / P_L|^2

for name, p in [("X", X), ("L", L)]:
    print(f"{name}:  Ac={p.Ac:.1f}  Bc={p.Bc:.1f}  Av={p.Av:.1f}  Bv={p.Bv:.1f}"
          f"  |  Abar={p.Abar:.1f}  Bbar={p.Bbar:.1f}  D={p.D:.1f}")

X:  Ac=12.3  Bc=9.5  Av=20.1  Bv=25.4  |  Abar=32.3  Bbar=15.9  D=503.2
L:  Ac=15.9  Bc=31.8  Av=5.4  Bv=3.7  |  Abar=21.3  Bbar=35.4  D=-114.1


## Regularised Quadrature

Both JDOS integrals have an integrable $1/\sqrt{}$ singularity, removed by substitution before calling `scipy.integrate.quad`:

**X-point** — singularity at $\mathcal{E}_{\max}$, substitute $t = \sqrt{\mathcal{E}_{\max} - \mathcal{E}}$:
$$\int_{\mathcal{E}_{\min}}^{\mathcal{E}_{\max}} \frac{[1-f(\mathcal{E})]\,d\mathcal{E}}{\sqrt{\mathcal{E}_{\max}-\mathcal{E}}} \;\longrightarrow\; \int_0^{\sqrt{\mathcal{E}_{\max}-\mathcal{E}_{\min}}} 2\,[1-f(\mathcal{E}_{\max}-t^2)]\,dt$$

**L-point** — singularity at $\mathcal{E}_{\min}$, substitute $t = \sqrt{\mathcal{E} - \mathcal{E}_{\min}}$:
$$\int_{\mathcal{E}_{\min}}^{\mathcal{E}_{\max}} \frac{[1-f(\mathcal{E})]\,d\mathcal{E}}{\sqrt{\mathcal{E}-\mathcal{E}_{\min}}} \;\longrightarrow\; \int_0^{\sqrt{\mathcal{E}_{\max}-\mathcal{E}_{\min}}} 2\,[1-f(\mathcal{E}_{\min}+t^2)]\,dt$$

In [5]:
def _F_rosei(E, hw, T):
    """Rosei Eq. (7): [1-f(E,T)]."""
    return 1.0 - fermi(E, T)

def _interband_integral(hw, T, p, F, is_L=False):
    """Regularised 1D JDOS integral (singularity removed by substitution)."""
    if hw <= p.Eg:
        return 0.0
    e_min, e_max = p.E_min(hw), p.E_max(hw)
    if e_max <= e_min:
        return 0.0
    t_max = np.sqrt(e_max - e_min)
    if is_L:
        integrand = lambda t: 2.0 * F(e_min + t**2, hw, T)
    else:
        integrand = lambda t: 2.0 * F(e_max - t**2, hw, T)
    result, _ = quad(integrand, 0, t_max)
    return p.prefactor() * result

## Interactive Fit to Rosei Fig. 8

Data digitised from Figure 8 of Guerrisi, Rosei & Winsemius (1975). Sliders adjust band parameters, Lorentzian broadening $\Gamma$, effective temperature $T_{\text{eff}}$, and overall scale. $R^2$ scores update live.

**Rosei's fitting parameters** (Sec. IV): $\mathcal{E}_g^X = 1.94$ eV, $\mathcal{E}_g^L = 2.45$ eV, $|P_X/P_L|^2 = 0.370$, $T_{\text{eff}} = 600$ K. Optical masses from Christensen & Seraphin (1971).

In [ ]:
%matplotlib tk
import os
from matplotlib.widgets import Slider, Button, CheckButtons
from scipy.signal import fftconvolve
from scipy.optimize import minimize

# ── Load Rosei Figure 8 data ────────────────────────────────────────────
_dir = os.path.dirname(os.path.abspath("__file__"))
data_X   = np.loadtxt(os.path.join(_dir, "a_e2_X.txt"),   delimiter=",", skiprows=1)
data_L   = np.loadtxt(os.path.join(_dir, "b_e2_L.txt"),   delimiter=",", skiprows=1)
data_tot = np.loadtxt(os.path.join(_dir, "c_e2_X_L.txt"), delimiter=",", skiprows=1)
for d in (data_X, data_L, data_tot):
    d[:] = d[d[:, 0].argsort()]

# ── Lorentzian broadening ────────────────────────────────────────────────
def _broaden(y, x, gamma):
    """Convolve y(x) with a normalised Lorentzian of HWHM gamma."""
    if gamma < 1e-4:
        return y.copy()
    dx = x[1] - x[0]
    x_k = x - x[len(x) // 2]
    kernel = (gamma / np.pi) / (x_k**2 + gamma**2) * dx
    return fftconvolve(y, kernel, mode='same')

# ── R² score ─────────────────────────────────────────────────────────────
def _r_squared(d_hw, d_e2, m_hw, m_e2):
    mi = np.interp(d_hw, m_hw, m_e2)
    ss_res = np.sum((d_e2 - mi)**2)
    ss_tot = np.sum((d_e2 - np.mean(d_e2))**2)
    return 1.0 - ss_res / ss_tot if ss_tot > 1e-30 else 0.0

# ── Defaults (Rosei Sec. IV) ────────────────────────────────────────────
DEFAULTS = dict(
    Eg_X=1.94, Eg_L=2.45,
    mc_perp_X=0.31, mc_par_X=0.40, mv_perp_X=0.19, mv_par_X=0.15,
    mc_perp_L=0.24, mc_par_L=0.12, mv_perp_L=0.70, mv_par_L=1.03,
    P_ratio_sq=0.370, T_val=600.0, Gamma=0.07,
)

# ── Compute ──────────────────────────────────────────────────────────────
hw_ext = np.linspace(1.0, 3.5, 300)          # extended grid (broadening padding)
hw_lo, hw_hi = 1.5, 3.0                       # display window
disp_mask = (hw_ext >= hw_lo) & (hw_ext <= hw_hi)
hw_disp = hw_ext[disp_mask]

def _compute(Eg_X, Eg_L, mc_perp_X, mc_par_X, mv_perp_X, mv_par_X,
             mc_perp_L, mc_par_L, mv_perp_L, mv_par_L, P_ratio_sq,
             T_val, Gamma):
    xp = XPointParams(Ac=C/mc_perp_X, Bc=C/mc_par_X,
                      Av=C/mv_perp_X, Bv=C/mv_par_X, Eg=Eg_X)
    lp = LPointParams(Ac=C/mc_perp_L, Bc=C/mc_par_L,
                      Av=C/mv_perp_L, Bv=C/mv_par_L, Eg=Eg_L)
    F = _F_rosei
    raw_t = np.empty(len(hw_ext)); raw_x = np.empty_like(raw_t); raw_l = np.empty_like(raw_t)
    for i, hw in enumerate(hw_ext):
        ix = N_X * P_ratio_sq * _interband_integral(hw, T_val, xp, F, is_L=False)
        il = N_L              * _interband_integral(hw, T_val, lp, F, is_L=True)
        raw_t[i] = (ix + il) / hw**2
        raw_x[i] = ix / hw**2
        raw_l[i] = il / hw**2
    return (_broaden(raw_t, hw_ext, Gamma)[disp_mask],
            _broaden(raw_x, hw_ext, Gamma)[disp_mask],
            _broaden(raw_l, hw_ext, Gamma)[disp_mask])

# Initial curves + auto-scale
tot0, cx0, cl0 = _compute(**DEFAULTS)
_mi = np.interp(data_tot[:, 0], hw_disp, tot0)
_mk = _mi > 1e-12
auto_scale = float(np.dot(data_tot[_mk, 1], _mi[_mk]) / np.dot(_mi[_mk], _mi[_mk]))
DEFAULTS["Scale"] = round(auto_scale, 2)
print(f"Auto-scale = {auto_scale:.1f}")

# ── Figure layout ────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 15))
ax  = fig.add_axes([0.08, 0.52, 0.88, 0.44])

# R² scores
r2 = dict(
    X   = _r_squared(data_X[:, 0],   data_X[:, 1],   hw_disp, auto_scale * cx0),
    L   = _r_squared(data_L[:, 0],   data_L[:, 1],   hw_disp, auto_scale * cl0),
    XL  = _r_squared(data_tot[:, 0], data_tot[:, 1], hw_disp, auto_scale * tot0))

# Data
scat = dict(
    XL = ax.plot(data_tot[:, 0], data_tot[:, 1], 'ko', ms=5, alpha=0.55, label='Rosei X+L')[0],
    X  = ax.plot(data_X[:, 0],  data_X[:, 1],  's', color='0.4', ms=4, alpha=0.55, label='Rosei X')[0],
    L  = ax.plot(data_L[:, 0],  data_L[:, 1],  '^', color='0.4', ms=4, alpha=0.55, label='Rosei L')[0])

# Model
line = dict(
    XL = ax.plot(hw_disp, auto_scale * tot0, 'k-',  lw=2.2, label=f'Model X+L  $R^2$={r2["XL"]:.4f}')[0],
    X  = ax.plot(hw_disp, auto_scale * cx0,  'k--', lw=1.4, label=f'Model X  $R^2$={r2["X"]:.4f}')[0],
    L  = ax.plot(hw_disp, auto_scale * cl0,  'k-.', lw=1.4, label=f'Model L  $R^2$={r2["L"]:.4f}')[0])

ax.set(xlabel=r'$\hbar\omega$ (eV)', ylabel=r'$\varepsilon_2$', xlim=(hw_lo, hw_hi), ylim=(0, 6))
ax.set_title("Rosei $\\varepsilon_2$ — Guerrisi, Rosei & Winsemius (1975), Fig. 8")
ax.grid(alpha=0.25)

# ── Visibility ───────────────────────────────────────────────────────────
_vis = dict(X=True, L=True, XL=True)

def _apply_vis():
    for k in _vis:
        line[k].set_visible(_vis[k])
        scat[k].set_visible(_vis[k])

def _rebuild_legend():
    h, lb = [], []
    for k, dlab, mlab in [('XL','Rosei X+L','Model X+L'),
                           ('X', 'Rosei X',  'Model X'),
                           ('L', 'Rosei L',  'Model L')]:
        if _vis[k]:
            h += [scat[k], line[k]]
            lb += [dlab, f'{mlab}  $R^2$={r2[k]:.4f}']
    ax.legend(h, lb, loc='upper left', fontsize=9, framealpha=0.9)

_rebuild_legend()

# ── Sliders ──────────────────────────────────────────────────────────────
specs = [
    ("Eg_X",       r"$E_g^X$ (eV)",       1.70, 2.10, DEFAULTS["Eg_X"],       0.01),
    ("mc_perp_X",  r"$m_{c\perp}^X$",     0.15, 0.60, DEFAULTS["mc_perp_X"],  0.01),
    ("mc_par_X",   r"$m_{c\parallel}^X$",  0.15, 0.60, DEFAULTS["mc_par_X"],   0.01),
    ("mv_perp_X",  r"$m_{v\perp}^X$",     0.10, 0.50, DEFAULTS["mv_perp_X"],  0.01),
    ("mv_par_X",   r"$m_{v\parallel}^X$",  0.05, 0.40, DEFAULTS["mv_par_X"],   0.01),
    ("Eg_L",       r"$E_g^L$ (eV)",       2.20, 2.60, DEFAULTS["Eg_L"],       0.01),
    ("mc_perp_L",  r"$m_{c\perp}^L$",     0.10, 0.50, DEFAULTS["mc_perp_L"],  0.01),
    ("mc_par_L",   r"$m_{c\parallel}^L$",  0.05, 0.30, DEFAULTS["mc_par_L"],   0.01),
    ("mv_perp_L",  r"$m_{v\perp}^L$",     0.30, 1.50, DEFAULTS["mv_perp_L"],  0.01),
    ("mv_par_L",   r"$m_{v\parallel}^L$",  0.50, 2.50, DEFAULTS["mv_par_L"],   0.01),
    ("P_ratio_sq", r"$|P_X/P_L|^2$",      0.10, 1.00, DEFAULTS["P_ratio_sq"], 0.01),
    ("Gamma",      r"$\Gamma$ (eV)",       0.00, 0.30, DEFAULTS["Gamma"],      0.005),
    ("Scale",      "Scale",  0.2*auto_scale, 5.0*auto_scale, auto_scale, 0.5),
    ("T_val",      r"$T_{\mathrm{eff}}$ (K)", 10, 1000, DEFAULTS["T_val"],    10),
]

sliders = {}
ns = len(specs)
sh, sg = 0.020, 0.030
sb = 0.47 - ns * sg

for i, (key, lab, lo, hi, v0, vs) in enumerate(specs):
    y = sb + (ns - 1 - i) * sg
    sl_ax = fig.add_axes([0.20, y, 0.62, sh])
    sliders[key] = Slider(sl_ax, lab, lo, hi, valinit=v0, valstep=vs)
    # Red reference marker at the C&S / Rosei literature value
    if key in DEFAULTS:
        sl_ax.axvline(DEFAULTS[key], color='red', ls='-', lw=1.2, alpha=0.7, zorder=5)

# ── Buttons row ──────────────────────────────────────────────────────────
ax_vis  = fig.add_axes([0.02, sb - 0.015, 0.12, 0.075])
chk_vis = CheckButtons(ax_vis, ['X', 'L', 'X+L'], [True, True, True])

ax_opt  = fig.add_axes([0.70, sb - 0.015, 0.11, 0.03])
btn_opt = Button(ax_opt, 'Optimize')

ax_rst  = fig.add_axes([0.84, sb - 0.015, 0.08, 0.03])
btn_rst = Button(ax_rst, 'Reset')

# ── Callbacks ────────────────────────────────────────────────────────────
def _update(_=None):
    s = {k: sl.val for k, sl in sliders.items()}
    scale = s.pop("Scale")
    tot, cx, cl = _compute(**s)
    line['XL'].set_ydata(scale * tot)
    line['X'].set_ydata(scale * cx)
    line['L'].set_ydata(scale * cl)
    r2['X']  = _r_squared(data_X[:, 0],   data_X[:, 1],   hw_disp, scale * cx)
    r2['L']  = _r_squared(data_L[:, 0],   data_L[:, 1],   hw_disp, scale * cl)
    r2['XL'] = _r_squared(data_tot[:, 0], data_tot[:, 1], hw_disp, scale * tot)
    _rebuild_legend()
    fig.canvas.draw_idle()

def _on_vis(label):
    k = 'XL' if label == 'X+L' else label
    _vis[k] = not _vis[k]
    _apply_vis(); _rebuild_legend(); fig.canvas.draw_idle()

def _on_reset(_):
    for k, sl in sliders.items():
        sl.set_val(DEFAULTS[k])

def _optimal_scale(tot, cx, cl):
    """Analytically solve for the least-squares optimal scale factor."""
    parts = []
    if _vis['XL']: parts.append((data_tot, tot))
    if _vis['X']:  parts.append((data_X, cx))
    if _vis['L']:  parts.append((data_L, cl))
    if not parts:  parts = [(data_tot, tot)]
    num = den = 0.0
    for d, m in parts:
        mi = np.interp(d[:, 0], hw_disp, m)
        num += np.dot(d[:, 1], mi)
        den += np.dot(mi, mi)
    return num / den if den > 1e-30 else auto_scale

def _on_optimize(_):
    """Nelder-Mead fit of all physics parameters against visible data.
    Scale is solved analytically at each cost-function evaluation."""
    fit_keys = [s[0] for s in specs if s[0] != 'Scale']
    bounds   = [(s[2], s[3]) for s in specs if s[0] != 'Scale']
    x0 = np.array([sliders[k].val for k in fit_keys])

    def cost(x):
        p = {k: float(np.clip(v, lo, hi))
             for k, v, (lo, hi) in zip(fit_keys, x, bounds)}
        try:
            tot, cx, cl = _compute(**p)
        except Exception:
            return 1e10
        scale = _optimal_scale(tot, cx, cl)
        parts = []
        if _vis['XL']: parts.append((data_tot, tot))
        if _vis['X']:  parts.append((data_X, cx))
        if _vis['L']:  parts.append((data_L, cl))
        if not parts:  parts = [(data_tot, tot)]
        return sum(np.sum((d[:, 1] - scale * np.interp(d[:, 0], hw_disp, m))**2)
                   for d, m in parts)

    ax.set_title("Optimizing\u2026")
    fig.canvas.draw_idle(); fig.canvas.flush_events()

    res = minimize(cost, x0, method='Nelder-Mead',
                   options={'maxiter': 3000, 'xatol': 1e-5, 'fatol': 1e-10,
                            'adaptive': True})

    # Apply optimised values to sliders
    for k, v, (lo, hi) in zip(fit_keys, res.x, bounds):
        sliders[k].set_val(float(np.clip(v, lo, hi)))

    # Set optimal scale
    s = {k: sliders[k].val for k in fit_keys}
    tot, cx, cl = _compute(**s)
    sliders['Scale'].set_val(round(_optimal_scale(tot, cx, cl), 2))

    ax.set_title("Rosei $\\varepsilon_2$ \u2014 Guerrisi, Rosei & Winsemius (1975), Fig. 8")
    print(f"Optimize: {res.message}, {res.nfev} evals, R\u00b2(XL)={r2['XL']:.5f}")

for sl in sliders.values():
    sl.on_changed(_update)
chk_vis.on_clicked(_on_vis)
btn_opt.on_clicked(_on_optimize)
btn_rst.on_clicked(_on_reset)

plt.show()

Auto-scale = 474.2


: 